# Task 2, Step 7 (Optional): Visual-Token Compression

The manual's optional experiment repeats the complete recipe with 49 visual tokens, averaging each $2\times2$ block of the $14\times14$ grid before the connector. It initializes a new connector and adapters and keeps seed, data, training budgets, and evaluation unchanged.

Set `RUN_COMPRESSION = True` in `kaggle_task2.ipynb` to run it. This notebook compares the two runs when both exist.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task2_vlm_grounding")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task2_vlm_grounding.src.notebook_utils import *
setup_notebook()

In [ ]:
if not os.path.exists(f"{RESULTS_DIR}/compressed_49/final_summary.csv"):
    display(Markdown("**The optional 49-token run has not been completed, so there is nothing to compare.**"))
else:
    tables = []
    for run, name in [("main", "196 tokens"), ("compressed_49", "49 tokens")]:
        table = pd.read_csv(f"{RESULTS_DIR}/{run}/final_summary.csv").assign(run=name)
        with open(f"{RESULTS_DIR}/{run}/summary.json") as f:
            info = json.load(f)
        table["training_min"], table["peak_gpu_mb"] = info["training_seconds"] / 60, info["peak_gpu_mb"]
        tables.append(table)
    comparison = pd.concat(tables)
    comparison.to_csv(f"{RESULTS_DIR}/token_compression_comparison.csv", index=False)
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
    for ax, suite in zip(axes[:2], ["grounding", "natural"]):
        sns.barplot(data=comparison[comparison["suite"] == suite], x="condition", y="score", hue="run", order=list(CONDITION_LABELS), ax=ax, edgecolor="black")
        ax.set(title=f"{suite}: score by condition", xlabel="", ylabel="%", ylim=(0, 100))
    resources = comparison.drop_duplicates("run").set_index("run")[["training_min", "peak_gpu_mb", "seconds_per_answer"]]
    (resources / resources.loc["196 tokens"]).T.plot(kind="bar", ax=axes[2], edgecolor="black", rot=15)
    axes[2].set(title="Resources relative to 196 tokens", ylabel="Ratio")
    save_figure("task2_7_token_compression")
    display(resources.round(3))